# 第 2 周额外周末练习 —— 自然语言转 SQL 原型

## 练习目标（理念）

把第 1 周「技术问答」思路升级成可用原型，综合运用第 2 周知识点，做成 **NL → SQL** 助手：

- **Gradio UI**：表单 + 示例，而不是只在笔记本里 `print`
- **流式（streaming）**：边生成边刷新 Markdown
- **System Prompt**：注入「SQL 数据库工程师」专业人设
- **多模型切换**：OpenAI / Anthropic / Gemini / 本地 Ollama（OpenAI 兼容 `base_url`）
- **奖励**：Tool Calling 在演示库上执行 SQL；Whisper 语音输入 + TTS 语音输出

## 和本课 Week 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 多提供商客户端 | `OpenAI(..., base_url=...)` 指向各家 |
| System Prompt | `system_prompt` 约束 SQL 生成格式 |
| Streaming | `generate_sql_stream` / `chat_with_tools` 里 `stream=True` |
| Tool / Function Calling | `execute_sql_query` + `tools` + `finish_reason == "tool_calls"` |
| 多模态 | Whisper 转写 + `gpt-4o-mini-tts` |

## 怎么跑

1. 从上到下依次运行单元格（Shift+Enter）
2. `.env` 至少配置 `OPENAI_API_KEY`；Anthropic / Google 可选；本地需 Ollama + `llama3.2`
3. 跑到 Gradio 那格会 `launch(inbrowser=True)`；用示例问题试流式生成，勾选工具可执行演示库 SQL


In [ ]:
# ========== 导入：环境、SQLite、OpenAI SDK、Gradio、笔记本展示 ==========

# 导入标准库 os：读环境变量里的各家 API Key
import os
# 导入标准库 json：解析 tool_call.function.arguments（JSON 字符串）
import json
# 导入标准库 sqlite3：演示库 sql_demo.db 的建表与执行
import sqlite3
# 从 dotenv 导入 load_dotenv：把 .env 读进进程环境
from dotenv import load_dotenv
# 从 openai 导入 OpenAI：云端与「OpenAI 兼容」端点共用同一 SDK
from openai import OpenAI
# 导入 gradio：搭 NL→SQL 的 Blocks UI
import gradio as gr 
# 从 IPython.display 导入 Markdown/display：笔记本内展示（本文件主要靠 Gradio）
from IPython.display import Markdown, display


In [ ]:
# ========== 加载环境变量并检查 API 密钥是否存在 ==========

# override=True：.env 里的值覆盖进程里已有同名变量
load_dotenv(override=True)
# 三家密钥：OpenAI 必用；Anthropic / Google 可选
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

# 只打印前缀，方便确认「读到了」又不过度暴露完整密钥
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set (optional)")


In [ ]:
# ========== 初始化多模型客户端 + MODELS 映射表 ==========

# 默认 OpenAI 云端客户端（读 OPENAI_API_KEY）
openai = OpenAI()

# 各家 OpenAI 兼容端点（URL 字符串保持原样，影响请求打到哪里）
anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
ollama_url = "http://localhost:11434/v1"

# 有密钥才建客户端；没有则置 None，后面 UI 选中会报「不可用」
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url) if anthropic_api_key else None
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url) if google_api_key else None
# 本地 Ollama：api_key 任意占位即可，真正鉴权在本地服务
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

# UI 下拉显示名 → {client, model_id}；model 字段是发给 API 的真实模型名
MODELS = {
    "GPT-4.1-mini": {"client": openai, "model": "gpt-4.1-mini"},
    "Claude Sonnet 4.5": {"client": anthropic, "model": "claude-sonnet-4-5-20250929"},
    "Gemini 2.5 Flash": {"client": gemini, "model": "gemini-2.5-flash-lite"},
    "Llama 3.2 (Local)": {"client": ollama, "model": "llama3.2"}
}

print("Model clients initialized successfully!")


In [ ]:
# ========== System Prompt：SQL 专家人设（发给模型的英文原文不翻译） ==========

# 约束：只按给定 schema 生成、用 ```sql 代码块、附简短解释、模糊时声明假设
system_prompt = """You are an expert SQL database engineer with deep knowledge of SQL syntax, optimization, and best practices.

Your role is to:
1. Generate accurate, efficient SQL queries from natural language questions
2. Use only the tables and columns provided in the database schema
3. Follow standard SQL syntax (PostgreSQL/SQLite compatible)
4. Provide clear explanations of your queries
5. Suggest optimizations when relevant

When responding:
- Return the SQL query in a fenced code block (```sql)
- Add a brief explanation of what the query does
- If the question is ambiguous, make reasonable assumptions and state them
- If a schema is not provided, generate generic SQL with common table/column names

Always prioritize correctness and clarity."""


In [ ]:
# ========== 流式 SQL 生成：NL + schema → 边收边 yield 完整文本 ==========

def generate_sql_stream(question, schema, model_name):
    """
    Generate SQL query from natural language with streaming response
    """
    # 从 MODELS 取客户端与真实 model id
    model_info = MODELS.get(model_name)
    # 未知名字或该提供商客户端为 None（缺密钥）→ 直接 yield 错误文案
    if not model_info or model_info["client"] is None:
        yield "Error: Selected model is not available. Please check your API keys."
        return
    
    client = model_info["client"]
    model = model_info["model"]
    
    # 有 schema 就拼进 user；没有就只发 Question
    user_content = f"Database Schema:\n{schema.strip()}\n\nQuestion: {question}" if schema.strip() else f"Question: {question}"
    
    # system 定专家角色；user 放 schema + 问题
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content}
    ]
    
    try:
        # stream=True：服务端推增量；客户端 for 循环拼 result
        stream = client.chat.completions.create(
            model=model,
            messages=messages,
            stream=True
        )
        
        result = ""
        for chunk in stream:
            # 每一小块 delta；None 当空串
            delta = chunk.choices[0].delta.content or ""
            result += delta
            # Gradio 需要「到目前为止的全文」，所以每次 yield 累积串
            yield result
            
    except Exception as e:
        # 网络/鉴权/模型名错误等，转成可见错误字符串
        yield f"Error: {str(e)}"


In [ ]:
# ========== 奖励工具：演示库 sql_demo.db + execute_sql_query ==========

# 演示数据库文件名（相对当前工作目录）
DB = "sql_demo.db"

def execute_sql_query(query):
    """
    Execute a SQL query against a demo database
    Returns the results as a formatted string
    """
    # 打日志：证明 tool 真的被调用了（flush 立刻刷出）
    print(f"TOOL CALLED: Executing SQL query", flush=True)
    try:
        # with 连接：离开块自动关闭
        with sqlite3.connect(DB) as conn:
            cursor = conn.cursor()
            # 执行模型传来的 SQL（演示用途；生产需严格校验）
            cursor.execute(query)
            results = cursor.fetchall()
            
            # 无行：仍算成功
            if not results:
                return "Query executed successfully. No results returned."
            
            # 列名来自 cursor.description
            column_names = [description[0] for description in cursor.description]
            result_str = f"Results ({len(results)} rows):\n\n"
            result_str += " | ".join(column_names) + "\n"
            result_str += "-" * (len(result_str) - 1) + "\n"
            
            # 逐行拼成可读表格文本
            for row in results:
                result_str += " | ".join(str(val) for val in row) + "\n"
            
            return result_str
    except Exception as e:
        return f"Error executing query: {str(e)}"

# ---------- 用示例数据初始化演示库（每次运行会 DROP 再 CREATE） ----------
def init_demo_db():
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        
        # 先删旧表，保证示例数据可重复演示
        cursor.execute('DROP TABLE IF EXISTS orders')
        cursor.execute('DROP TABLE IF EXISTS customers')
        cursor.execute('DROP TABLE IF EXISTS products')
        
        # customers：客户主数据
        cursor.execute('''
            CREATE TABLE customers (
                id INTEGER PRIMARY KEY,
                name TEXT,
                email TEXT,
                created_at DATE
            )
        ''')
        
        # orders：订单，外键指向 customers
        cursor.execute('''
            CREATE TABLE orders (
                id INTEGER PRIMARY KEY,
                customer_id INTEGER,
                total REAL,
                order_date DATE,
                FOREIGN KEY (customer_id) REFERENCES customers(id)
            )
        ''')
        
        # products：商品价目
        cursor.execute('''
            CREATE TABLE products (
                id INTEGER PRIMARY KEY,
                name TEXT,
                price REAL
            )
        ''')
        
        # 插入示例客户
        cursor.executemany('INSERT INTO customers VALUES (?, ?, ?, ?)', [
            (1, 'Alice Johnson', 'alice@email.com', '2025-01-15'),
            (2, 'Bob Smith', 'bob@email.com', '2025-12-20'),
            (3, 'Carol White', 'carol@email.com', '2026-02-01'),
            (4, 'David Brown', 'david@email.com', '2026-02-15')
        ])
        
        # 插入示例订单
        cursor.executemany('INSERT INTO orders VALUES (?, ?, ?, ?)', [
            (1, 1, 150.00, '2026-02-01'),
            (2, 1, 200.00, '2026-02-15'),
            (3, 2, 75.00, '2025-12-25'),
            (4, 3, 300.00, '2026-02-20'),
            (5, 4, 450.00, '2026-02-28')
        ])
        
        # 插入示例商品
        cursor.executemany('INSERT INTO products VALUES (?, ?, ?)', [
            (1, 'Laptop', 999.99),
            (2, 'Mouse', 29.99),
            (3, 'Keyboard', 79.99),
            (4, 'Monitor', 299.99)
        ])
        
        conn.commit()
    print("Demo database initialized with sample data!")

# 定义完立刻初始化，后面 tool 调用才有数据
init_demo_db()


In [ ]:
# ========== Tool Schema：告诉模型可以调用 execute_sql_query ==========

# OpenAI tools 格式：name / description / parameters（JSON Schema）
# description 与参数说明是给模型看的英文，保持原样
execute_sql_function = {
    "name": "execute_sql_query",
    "description": "Execute a SQL query against the demo database and return the results. Use this when the user wants to run or test a generated SQL query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The SQL query to execute"
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

# chat.completions 的 tools= 参数需要外层 type=function 包装
tools = [{"type": "function", "function": execute_sql_function}]


In [ ]:
# ========== 带工具循环的对话：可执行 SQL，或退回纯流式生成 ==========

def handle_tool_calls(message):
    """Handle tool calls from the LLM"""
    responses = []
    # 一条 assistant message 可能带多个 tool_calls
    for tool_call in message.tool_calls:
        # 只实现本练习注册的那一个函数名
        if tool_call.function.name == "execute_sql_query":
            # arguments 是 JSON 字符串 → dict
            arguments = json.loads(tool_call.function.arguments)
            query = arguments.get('query')
            # 真正跑 SQLite
            result = execute_sql_query(query)
            # 按协议回传 role=tool，并带上 tool_call_id 对齐
            responses.append({
                "role": "tool",
                "content": result,
                "tool_call_id": tool_call.id
            })
    return responses

def chat_with_tools(question, schema, model_name, enable_tools):
    """
    Chat function that supports tool calling for SQL execution
    """
    model_info = MODELS.get(model_name)
    if not model_info or model_info["client"] is None:
        yield "Error: Selected model is not available. Please check your API keys."
        return
    
    client = model_info["client"]
    model = model_info["model"]
    
    # 与纯流式路径相同的 user 拼装
    user_content = f"Database Schema:\n{schema.strip()}\n\nQuestion: {question}" if schema.strip() else f"Question: {question}"
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content}
    ]
    
    try:
        if enable_tools:
            # 工具路径：非流式，便于读 finish_reason / tool_calls
            response = client.chat.completions.create(
                model=model,
                messages=messages,
                tools=tools,
                stream=False
            )
            
            # 标准 tool loop：只要还在要工具，就执行 → 追加 → 再请求
            while response.choices[0].finish_reason == "tool_calls":
                message = response.choices[0].message
                tool_responses = handle_tool_calls(message)
                messages.append(message)
                messages.extend(tool_responses)
                response = client.chat.completions.create(
                    model=model,
                    messages=messages,
                    tools=tools,
                    stream=False
                )
            
            # 最终自然语言 + SQL 解释
            yield response.choices[0].message.content
        else:
            # 关闭工具：回到流式打字机体验
            stream = client.chat.completions.create(
                model=model,
                messages=messages,
                stream=True
            )
            
            result = ""
            for chunk in stream:
                delta = chunk.choices[0].delta.content or ""
                result += delta
                yield result
            
    except Exception as e:
        yield f"Error: {str(e)}"


In [ ]:
# ========== 奖励：音频输入（Whisper）与音频输出（TTS） ==========

def transcribe_audio(audio_file):
    """Convert audio input to text using Whisper"""
    # Gradio 未录音时是 None
    if audio_file is None:
        return ""
    
    try:
        # 以二进制打开临时音频文件，交给 Whisper
        with open(audio_file, "rb") as f:
            transcript = openai.audio.transcriptions.create(
                model="whisper-1",
                file=f
            )
        # 返回转写文本，供后续当 question
        return transcript.text
    except Exception as e:
        print(f"Audio transcription error: {str(e)}")
        return ""

def text_to_speech(text):
    """Convert text response to speech"""
    try:
        # gpt-4o-mini-tts + alloy 音色；返回音频字节
        response = openai.audio.speech.create(
            model="gpt-4o-mini-tts",
            voice="alloy",
            input=text
        )
        return response.content
    except Exception as e:
        print(f"Text-to-speech error: {str(e)}")
        return None


In [ ]:
# ========== Gradio UI：语音/文本输入 + 工具开关 + 流式输出 + TTS ==========

def process_query(audio_input, text_input, schema, model_name, enable_tools, enable_audio_output):
    """
    Main processing function that handles both audio and text input
    """
    # 默认用文本框内容
    question = text_input
    
    # 若有麦克风录音：Whisper 转写后覆盖 question
    if audio_input is not None:
        transcribed = transcribe_audio(audio_input)
        if transcribed:
            question = transcribed
    
    # 两者都空 → 提示用户
    if not question.strip():
        yield "Please provide a question either via text or audio.", None
        return
    
    # 先流式更新 Markdown；音频先传 None，避免半成品朗读
    result = ""
    for chunk in chat_with_tools(question, schema, model_name, enable_tools):
        result = chunk
        yield chunk, None
    
    # 勾选语音输出时，对最终全文做 TTS
    if enable_audio_output:
        audio = text_to_speech(result)
        yield result, audio
    else:
        yield result, None

# 默认 schema：与演示库三张表对齐（字符串保持原样）
default_schema = """Table: customers (id, name, email, created_at)
Table: orders (id, customer_id, total, order_date)
Table: products (id, name, price)"""

# 构建 Blocks：左栏输入，右栏结果
with gr.Blocks(title="SQL Query Generator") as demo:
    gr.Markdown("# 🗄️ Natural Language to SQL Query Generator")
    gr.Markdown("Ask questions in natural language and get SQL queries generated by AI. Optionally use your voice!")
    
    with gr.Row():
        with gr.Column(scale=2):
            # 麦克风 → 临时文件路径，交给 Whisper
            audio_input = gr.Audio(
                sources=["microphone"],
                type="filepath",
                label="🎤 Voice Input (Optional)"
            )
            text_input = gr.Textbox(
                label="💬 Text Input",
                placeholder="e.g., List all customers who placed an order in the last 30 days",
                lines=3
            )
            schema_input = gr.Textbox(
                label="📊 Database Schema (Optional)",
                value=default_schema,
                lines=5,
                placeholder="Describe your database tables and columns"
            )
            
            with gr.Row():
                # 下拉选项来自 MODELS 的 key
                model_selector = gr.Dropdown(
                    choices=list(MODELS.keys()),
                    value="GPT-4.1-mini",
                    label="🤖 Select Model"
                )
                # 是否走 tool calling 执行演示库
                enable_tools = gr.Checkbox(
                    label="🔧 Enable SQL Execution",
                    value=False,
                    info="Allow the AI to execute queries on demo database"
                )
                # 是否对最终回答做 TTS
                enable_audio = gr.Checkbox(
                    label="🔊 Audio Response",
                    value=False,
                    info="Get spoken response"
                )
            
            submit_btn = gr.Button("Generate SQL Query", variant="primary")
        
        with gr.Column(scale=3):
            output = gr.Markdown(label="📝 Generated SQL & Explanation")
            audio_output = gr.Audio(label="🔊 Audio Response", autoplay=True)
    
    # 一键填充示例（含模型名、是否开工具）；字符串保持原样
    gr.Examples(
        examples=[
            [None, "List all customers who placed an order in the last 30 days, with their total spend", default_schema, "GPT-4.1-mini", False, False],
            [None, "Show me the top 5 most expensive products", default_schema, "GPT-4.1-mini", False, False],
            [None, "Find customers who haven't placed any orders", default_schema, "Claude Sonnet 4.5", False, False],
            [None, "What is the average order value per customer?", default_schema, "GPT-4.1-mini", True, False],
        ],
        inputs=[audio_input, text_input, schema_input, model_selector, enable_tools, enable_audio]
    )
    
    # 按钮点击 → process_query
    submit_btn.click(
        fn=process_query,
        inputs=[audio_input, text_input, schema_input, model_selector, enable_tools, enable_audio],
        outputs=[output, audio_output]
    )
    
    # 文本框回车同样提交
    text_input.submit(
        fn=process_query,
        inputs=[audio_input, text_input, schema_input, model_selector, enable_tools, enable_audio],
        outputs=[output, audio_output]
    )

# 启动 Gradio，并尝试在浏览器打开
demo.launch(inbrowser=True)


In [ ]:
# ========== 替代方案：不启工具的简单流式包装（下方 Interface 仍注释掉） ==========

def simple_generate(question, schema, model_name):
    """Simple streaming SQL generation without tool calling"""
    # 直接把生成器委托给 generate_sql_stream（无 tool loop、无音频）
    yield from generate_sql_stream(question, schema, model_name)

# 取消下面的注释即可使用更简单的接口，无需调用工具：
# （以下为历史注释块，保持原样不改写，避免动到备用 UI 草稿）

# Question_input = gr.Textbox(
# 标签=“您的问题：”，
# placeholder="例如，显示上周的所有订单",
# 行=3
# )
# schema_input = gr.Textbox(
# label="数据库架构：",
# 值=默认模式，
# 行=5
# )
# model_selector = gr.Dropdown(
# 选择=列表(MODELS.keys()),
# 值=“GPT-4.1-mini”，
# 标签=“选择型号”
# )
# 输出 = gr.Markdown(label="生成的 SQL:")

# 视图 = gr. 接口(
# fn=简单生成，
# title="SQL 查询生成器",
# 输入=[问题输入、模式输入、模型选择器]、
# 输出=[输出],
# 例子=[
# [“列出所有客户”，default_schema，“GPT-4.1-mini”]，
# [“显示上个月的订单”，default_schema，“克劳德十四行诗 4.5”]
#     ],
# flagging_mode =“从不”
# )
# view.launch(inbrowser=True)


# 如何使用此应用（教学说明）

## 已实现的功能

### 核心要求
1. **Gradio UI**：左右分栏，文本/语音输入 + 结果展示
2. **流式传输**：未开工具时边生成边刷新 Markdown
3. **系统提示**：`system_prompt` 固定 SQL 专家角色与输出格式
4. **模型选择**：GPT-4.1-mini、Claude Sonnet 4.5、Gemini 2.5 Flash、Llama 3.2（本地）

### 奖励功能
5. **工具调用**：勾选后走 `execute_sql_query`，在 `sql_demo.db` 上跑 SQL
6. **音频输入**：麦克风 → Whisper（`whisper-1`）转写
7. **音频输出**：最终回答 → `gpt-4o-mini-tts` 朗读

## 使用步骤

### 基本用法
1. 用自然语言输入问题（或用麦克风）
2. 按需改 Database Schema 文本框
3. 选择模型
4. 点击 Generate SQL Query

### 高级开关
- **Enable SQL Execution**：允许模型通过 tool 在演示库执行查询
- **Audio Response**：对最终全文做 TTS
- **Voice Input**：点麦克风图标说话提问

## 示例问题（可对照右侧 Examples）
- 列出过去 30 天下过订单的客户及总消费
- 平均订单金额是多少？
- 显示没有订单的客户
- 找到最贵的产品

## 演示库表结构
- **customers**：id, name, email, created_at
- **orders**：id, customer_id, total, order_date
- **products**：id, name, price

---

**注意**：`.env` 至少配置 `OPENAI_API_KEY`；Anthropic / Google 可选；本地 Llama 需 Ollama 已拉取 `llama3.2`。
